In [ ]:
import os
import json

In [ ]:
BIOPORTAL_BASE = os.environ.get("BIOPORTAL_BASE")
BIOPORTAL_KEY = os.environ.get("BIOPORTAL_API_KEY")
QWEN_SERVER_URL = os.environ.get("QWEN_SERVER_URL")
EMBED_MODEL = os.environ.get("EMBED_MODEL")

# EDA

In [ ]:
import requests

BASE = "https://data.bioontology.org"
endpoint = "/search" # annotate?
params = {
    "q": "heart attack",
    "ontologies": "NCIT",
    "require_exact_match": "true",
    "include": "prefLabel,definition",
    "apikey": BIOPORTAL_KEY
}

response = requests.get(BASE + endpoint, params=params)
if response.status_code == 200:
    data = response.json()
    # do something with `data`
else:
    print(f"Error: {response.status_code} - {response.text}")


In [24]:
data

{'page': 1,
 'pageCount': 1,
 'totalCount': 1,
 'prevPage': None,
 'nextPage': None,
 'links': {'nextPage': None, 'prevPage': None},
 'collection': [{'prefLabel': 'Myocardial Infarction',
   'definition': ['Gross necrosis of the myocardium, as a result of interruption of the blood supply to the area, as in coronary thrombosis.'],
   '@id': 'http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C27996',
   '@type': 'http://www.w3.org/2002/07/owl#Class',
   'links': {'self': 'https://data.bioontology.org/ontologies/NCIT/classes/http%3A%2F%2Fncicb.nci.nih.gov%2Fxml%2Fowl%2FEVS%2FThesaurus.owl%23C27996',
    'ontology': 'https://data.bioontology.org/ontologies/NCIT',
    'children': 'https://data.bioontology.org/ontologies/NCIT/classes/http%3A%2F%2Fncicb.nci.nih.gov%2Fxml%2Fowl%2FEVS%2FThesaurus.owl%23C27996/children',
    'parents': 'https://data.bioontology.org/ontologies/NCIT/classes/http%3A%2F%2Fncicb.nci.nih.gov%2Fxml%2Fowl%2FEVS%2FThesaurus.owl%23C27996/parents',
    'descendants': 'htt

Use annotator

In [ ]:
text = "Melanoma"

longest_only = True
expand_mappings = False
minimum_match_length = 0
pagesize = 200

headers = {"Authorization": f"apikey token={BIOPORTAL_KEY}"}
params = {
    "text": text,
    "longest_only": "true" if longest_only else "false",
    "expand_mappings": "true" if expand_mappings else "false",
    "minimum_match_length": str(minimum_match_length),
    "pagesize": str(pagesize)
}

resp = requests.get(f"{BASE}/annotator", headers=headers, params=params, timeout=30)
resp.raise_for_status()
data = resp.json()

items = data if isinstance(data, list) else data.get("annotations") or data.get("collection") \
        or data.get("matches") or data.get("results")

In [60]:
items

[{'annotatedClass': {'@id': 'http://purl.obolibrary.org/obo/OMIT_0009611',
   '@type': 'http://www.w3.org/2002/07/owl#Class',
   'links': {'self': 'https://data.bioontology.org/ontologies/OMIT/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FOMIT_0009611',
    'ontology': 'https://data.bioontology.org/ontologies/OMIT',
    'children': 'https://data.bioontology.org/ontologies/OMIT/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FOMIT_0009611/children',
    'parents': 'https://data.bioontology.org/ontologies/OMIT/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FOMIT_0009611/parents',
    'descendants': 'https://data.bioontology.org/ontologies/OMIT/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FOMIT_0009611/descendants',
    'ancestors': 'https://data.bioontology.org/ontologies/OMIT/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FOMIT_0009611/ancestors',
    'instances': 'https://data.bioontology.org/ontologies/OMIT/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FOMIT_0009611/instance

# RAG


Here, you still use BioPortal — but as a data source, not a search engine:

- Fetch candidate ontology entries (labels, synonyms, definitions).
- Encode them into a semantic vector space using SentenceTransformer.
- Use FAISS for semantic retrieval of the most relevant entries.
- Then feed only the top few to your LLM (Qwen) for reasoning.
- So it’s RAG over ontologies, not RAG over documents.

#### ✅ Pros:
- Semantic retrieval: “heart attack” finds “myocardial infarction” even if the text differs.
- Fast: local FAISS queries after initial caching.
- Extendable: you can merge multiple ontologies (SNOMED, FMA, GO, etc.) into one searchable space.
- Custom control: tune embedding model, scoring, context formatting.

####  ❌ Cons:
- Setup complexity (FAISS, embeddings, storage).
- Initial indexing cost.
- Needs local compute.

In [ ]:
from utils.rag import rag_match_for_term_annotator, rag_match_for_term_search

Example usage search

In [70]:
if __name__ == "__main__":
    # source_label = "melanoma"
    source_label = "heart attack"
    # source_def = "An acute condition of necrosis of heart muscle due to ischemia (heart attack)."
    source_def = None
    print("Running RAG match using Qwen server...")
    try:
        results = rag_match_for_term_search(source_label, source_def, ontologies="NCIT", top_k=5)
        print(json.dumps(results, indent=2))
    except Exception as e:
        print("Error:", e)


Running RAG match using Qwen server...
[
  {
    "candidate_uri": "http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C27996",
    "candidate_label": "Myocardial Infarction",
    "mapping_type": "EXACT",
    "confidence": 0.44,
    "reason": "Exact match with the source label 'heart attack' as both refer to the same medical condition."
  },
  {
    "candidate_uri": "http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C92620",
    "candidate_label": "Panic Attack",
    "mapping_type": "NO_MATCH",
    "confidence": 0.0,
    "reason": "No match with the source concept 'heart attack'."
  },
  {
    "candidate_uri": "http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C125857",
    "candidate_label": "PHQ Screener Version - Anxiety Attack Chest Pain",
    "mapping_type": "NO_MATCH",
    "confidence": 0.0,
    "reason": "No match with the source concept 'heart attack'."
  },
  {
    "candidate_uri": "http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C125856",
    "candidate_label": "PHQ Scree

test for annotator

In [75]:
if __name__ == "__main__":
    source_label = "melanoma"
    # source_label = "heart attack"
    # source_def = "An acute condition of necrosis of heart muscle due to ischemia (heart attack)."
    source_def = None
    print("Running RAG match using Qwen server...")
    try:
        results = rag_match_for_term_annotator(source_label, source_def, top_k=5)
        print(json.dumps(results, indent=2))
    except Exception as e:
        print("Error:", e)


Running RAG match using Qwen server...
[
  {
    "candidate_uri": null,
    "candidate_label": "melanoma",
    "mapping_type": "EXACT",
    "confidence": 0.225,
    "reason": "Exact match with source label"
  },
  {
    "candidate_uri": null,
    "candidate_label": "melanoma",
    "mapping_type": "EXACT",
    "confidence": 0.225,
    "reason": "Exact match with source label"
  },
  {
    "candidate_uri": null,
    "candidate_label": "melanoma",
    "mapping_type": "EXACT",
    "confidence": 0.225,
    "reason": "Exact match with source label"
  },
  {
    "candidate_uri": null,
    "candidate_label": "melanoma",
    "mapping_type": "EXACT",
    "confidence": 0.225,
    "reason": "Exact match with source label"
  },
  {
    "candidate_uri": null,
    "candidate_label": "melanoma",
    "mapping_type": "EXACT",
    "confidence": 0.225,
    "reason": "Exact match with source label"
  }
]


# Matching pipeline